# PROJEKT : <span style="margin-left:40px;"> Jerzy Marcin Urbański</span>

##### TEMAT:
## Analiza Jakości Powietrza Dla Wybranych Lokalizacji Oraz Okresów

Importy Bibliotek wraz z ich opisem 

Sekcja bibliotek

In [29]:
# ### pandas

# Biblioteka do pracy z danymi tabelarycznymi (DataFrame), umożliwia łatwe filtrowanie, grupowanie i przekształcanie danych. 
# Używana głównie do analizy danych, czyszczenia danych oraz wczytywania/zapisywania plików (CSV, Excel, SQL).
import pandas as pd

# ### numpy
# Podstawowa biblioteka do obliczeń numerycznych w Pythonie, oferująca wydajne operacje na tablicach wielowymiarowych. 
# Wykorzystywana do obliczeń matematycznych, statystycznych oraz jako fundament dla innych bibliotek analitycznych.
import numpy as np

# ### os
# Biblioteka standardowa do interakcji z systemem operacyjnym, umożliwia pracę z plikami i katalogami oraz zmiennymi środowiskowymi. 
#Używana m.in. do automatyzacji operacji na plikach i zarządzania ścieżkami.
import os

# ### matplotlib
# Biblioteka do tworzenia wykresów i wizualizacji danych w Pythonie. Pozwala generować różne typy wykresów (liniowe, słupkowe, histogramy) 
# oraz dostosowywać ich wygląd.
import matplotlib.pyplot as plt

# ### seaborn
# Rozszerzenie matplotlib, które upraszcza tworzenie estetycznych i statystycznych wizualizacji danych. 
# Umożliwia szybkie generowanie wykresów takich jak heatmapy, wykresy rozkładu czy zależności między zmiennymi.
import seaborn as sns

# ### ydata_profiling
# Biblioteka do automatycznej analizy eksploracyjnej danych (EDA), generująca szczegółowe raporty o zbiorze danych. 
# Pokazuje statystyki, brakujące wartości, korelacje i potencjalne problemy w danych.
from ydata_profiling import ProfileReport

# ### IPython.display
# Moduł umożliwiający wyświetlanie różnych typów treści w notebookach (HTML, obrazy, wideo). 
#  Używany do wzbogacania prezentacji wyników analiz i tworzenia interaktywnych raportów.
from IPython.display import IFrame
from pathlib import Path

Sekcja bibliotek ML

In [31]:
# import....

## STEP 1. Wczytanie danych + transformacja datasetów przygotowanie do głównego połaczenia.
Import danych z plików csv , ostateczne sprawdznie oraz potrzebna agregacja do ostatecznej wersji.

### 1.1. - DANE - weather_pz/ld_2023/2024

##### 1.1.1. Import wszyskich plików do odpowiednich zmiennych 


In [35]:
# import wszyskich plików do odpowiednich zmiennych 

w_ld_2023 = pd.read_csv('/Users/marcinurbanski/Desktop/jmu_repo/1_projekt_air_polution_repo/0_ready_raw_data/weather_ld_2023.csv')
w_ld_2024 = pd.read_csv('/Users/marcinurbanski/Desktop/jmu_repo/1_projekt_air_polution_repo/0_ready_raw_data/weather_ld_2024.csv')
w_pz_2023 = pd.read_csv('/Users/marcinurbanski/Desktop/jmu_repo/1_projekt_air_polution_repo/0_ready_raw_data/weather_pz_2023.csv')
w_pz_2024 = pd.read_csv('/Users/marcinurbanski/Desktop/jmu_repo/1_projekt_air_polution_repo/0_ready_raw_data/weather_pz_2024.csv')

In [36]:
# szybkie sprawdznie 

# display(w_ld_2023.head(2))
# display(w_ld_2024.head(2))
# display(w_pz_2023.head(2))
# display(w_pz_2024.head(2))

#### 1.1.2. Połaczenie w jedne DataFrame

In [38]:
w_df = pd.concat([w_ld_2023,w_ld_2024,w_pz_2023,w_pz_2024], axis=0 , ignore_index=True)

#### 1.1.2. - Agregacja danych do postaci uniwersalnego DataFrame 

In [40]:
# przypisanie odpowiedniego id miasta [city] wg współrzędnych

w_df['city'] = w_df['longitude'].map({17.0 : 'PZ' , 19.5 : 'LD'})

In [41]:
# Selekscja kolumn do usunięcia 

# print(w_ld_2004['step'].nunique()) # kolumna ma jeden wynik -> usunać
# print('-'*140)
# print(w_ld_2004['surface'].nunique()) # kolumna ma jeden wynik -> usunać
# print('-'*140)
# print(w_ld_2004[w_ld_2004["time"] != w_ld_2004["valid_time"]]) # kolumna 'valid_time' powtrza wartosci 'time' -> usunać 
# print('-'*140)
# print(w_ld_2004['number'].nunique())

In [42]:
# usunięcie niepotrzebnych kolumn ['Unnamed: 0','latitude','longitude','step','surface','valid_time']

w_df = w_df.drop(['Unnamed: 0','latitude','longitude','step','surface','valid_time','number'], axis=1)

In [43]:
# Zamiana typu danych object z kolumny 'time' na date_time

w_df['time'] = pd.to_datetime(w_df['time'])

Obliczenie prędkości wiatru ze zmiennych u10 & v10 ze wzoru tj z równania pitagorasa
- zmienna u10 => prędkość wiatru zachodniego 
- zmienna v10 => prędkość wiartu północnego 

$$
v_w = \sqrt{u_{10}^2 + v_{10}^2}
$$


In [45]:
# przeliczenie prędkości wiatru 'v_w' i dodanie jako nowa kolumna z tymi wartościami

w_df['v_w'] = np.sqrt(w_df['u10']**2 + w_df['v10']**2)

In [46]:
# ułożenie kolejnosci kolumn [info data | zmienne pogodowe] 

col_order = ['city','time','u10', 'v10','v_w','t2m','blh','sp','tcc']
w_df = w_df[col_order]

#### 1.1.4. PODSUMOWANIE PKT. 1.1 DANE POGOWE 
OPIS ZMIENNYCH 
- 'city' -> MIASTO ZAPISANE W SYMOBOLEM
    - PZ = Poznań
    - LD = Łódź
- 'time'............ DATA GODZINA
- 'u10' -> predkosc wiatru z kierunku / jednostko = [ m/s ] 
- 'v10' -> prędkość wiatru z kierunku / jednostka = [ m/s ]
- 'v_w' -> rzeczywista prędkość wiatru / jednostka = [ m/s ]
- 't2m' -> temperatura powietrze 2m nad poziomem powierzchni / jednostka [ Kelwin ]
- 'blh' -> WYSOKOŚĆ WARSTWY GRANICZNEJ / jednostka [ m ]
- 'sp' -> CIŚNIENIE / jednostka [ Pa ]
- 'tcc' -> CAŁKOWITE ZACHMURZENIE / jednostk [ 0-1 ] [ % ]

    

## 1.2. TABELE CAŁKOWITE OPADY - total_precipition

##### 1.2.1. Import wszyskich plików do odpowiednich zmiennych 

In [50]:
tp_ld_2004 = pd.read_csv('/Users/marcinurbanski/Desktop/jmu_repo/1_projekt_air_polution_repo/0_ready_raw_data/total_precipitation_ld_2023.csv')
tp_ld_2024 = pd.read_csv('/Users/marcinurbanski/Desktop/jmu_repo/1_projekt_air_polution_repo/0_ready_raw_data/total_precipitation_ld_2024.csv')
tp_pz_2004 = pd.read_csv('/Users/marcinurbanski/Desktop/jmu_repo/1_projekt_air_polution_repo/0_ready_raw_data/total_precipitation_pz_2023.csv')
tp_pz_2024 = pd.read_csv('/Users/marcinurbanski/Desktop/jmu_repo/1_projekt_air_polution_repo/0_ready_raw_data/total_precipitation_pz_2024.csv')

In [51]:
# szybkie sprawdznie 

# print(tp_ld_2004.info())
# print('^'*140)
# display(tp_ld_2004.head(10))
# display(tp_ld_2024.head(2))
# display(tp_pz_2004.head(2))
# display(tp_pz_2024.head(2))

####  1.2.2. Połaczenie wszystkich Data Frame w jeden pogodowy 

In [53]:
tp_df = pd.concat([tp_ld_2004,tp_ld_2024,tp_pz_2004,tp_pz_2024], axis=0 , ignore_index=True).reset_index()

#### 1.2.3. - Agregacja danych do postaci uniwersalnego DataFrame 

In [55]:
# usuwamy zbedne kolumny, nie potrzebne do analizy i dalszej agregacji 

tp_df = tp_df.drop(['Unnamed: 0','time','step','number','surface','index'], axis=1) 

In [56]:
# Zamiana kolumn z data i czasem na typ danych date_time

tp_df['valid_time'] =  tp_df['valid_time'].apply(pd.to_datetime)

# Usuwamy niepotrzebne punkty pomiarowe z innych godzin 'valid_time' niż 00:00, 06:00, 12:00, 18:00 każdego dnia

tp_df = tp_df[tp_df['valid_time'].dt.hour.isin([0, 6, 12, 18])]

# zamieniamy nazwę valid_time na time , aby pasowała do poprzedniego DataFrame dane pogodowe i mozliwosci ich połaczenia

tp_df = tp_df.rename(columns={'valid_time':'time'})

In [57]:
# sprawedzenie wartosci longitude dla wybranych miast 
tp_df['longitude'].unique()

array([19.5, 17. ])

In [58]:
# przypisanie skrótu miasta w nowej kolumnie 'city' + usuniecie niepotrzebnej kolu

tp_df['city'] = tp_df['longitude'].map({17.0 : 'PZ' , 19.5 : 'LD'})

In [59]:
# usuwamy zbedne kolumny lognitude i 

tp_df = tp_df.drop(['longitude','latitude'], axis=1) 

#### 1.2.4. PODSUMOWANIE PKT. 1.2. CAŁKOWITE OPADY  
OPIS ZMIENNYCH 
- 'PZ' -> MIASTO ZAPISANE W SYMOBOLEM
    - PZ = Poznań
    - LD = Łódź
- 'time'............ DATA GODZINA
- 'tp' -> CAŁKOWITE OPADY / jednostka [ m ] - skumulowany opad 

## 1.3. TABELE INWERSJE - inwesrsja [..].csv

#### 1.3.1. Import wszystich plikow do odpowiednich zmienych 

In [63]:
i_ld_2023 = pd.read_csv('/Users/marcinurbanski/Desktop/jmu_repo/1_projekt_air_polution_repo/0_ready_raw_data/inwersja_ld_2023.csv')
i_ld_2024 = pd.read_csv('/Users/marcinurbanski/Desktop/jmu_repo/1_projekt_air_polution_repo/0_ready_raw_data/inwersja_ld_2024.csv')
i_pz_2023 = pd.read_csv('/Users/marcinurbanski/Desktop/jmu_repo/1_projekt_air_polution_repo/0_ready_raw_data/inwersja_pz_2023.csv')
i_pz_2024 = pd.read_csv('/Users/marcinurbanski/Desktop/jmu_repo/1_projekt_air_polution_repo/0_ready_raw_data/inwersja_pz_2024.csv')

In [64]:
# szybkie sprawdzenie 

# print(i_ld_2004.info())
# print('^'*140)
# display(i_ld_2023.head(2))
# display(i_ld_2024.head(2))
# display(i_pz_2023.loc[i_pz_2023['time']=='2023-04-23 18:00:00'])
# display(i_pz_2024.head(2))

#### 1.3.2. Łaczymy wszyskie tabele do jednego DataFrame

In [66]:
i_df = pd.concat([i_ld_2023, i_ld_2024, i_pz_2023, i_pz_2024],  axis=0, ignore_index=True)

#### 1.3.3. Agregacja danych do postaci uniwersalnego datasetu.

In [68]:
# przypisanie skrótu miasta w nowej kolumnie 'city' + usuniecie niepotrzebnej kolu

i_df['city'] = i_df['longitude'].map({17.0 : 'PZ' , 19.5 : 'LD'})

In [69]:
# Zamiana typu danych kolumny 'time' na date_time

i_df['time'] = pd.to_datetime(i_df['time'])

In [70]:
## USTALENIE INWERSJI 0/1 NA PODSTAWIE RÓŻNICY TEMPERATURY NA WYSOKOŚĆI Z CIŚNIENEIM 1000 A 925

# Rozdzielam na dwa DataFrame 
df_925 = i_df[i_df['isobaricInhPa'] == 925][['time','t','city']].rename(columns = {'t': 't_925'})
df_1000 = i_df[i_df['isobaricInhPa'] == 1000][['time','t','city']].rename(columns = {'t': 't_1000'})

# Łaczymy oba df, w jeden do obliczenia różnicy
i_df_delta = pd.merge(df_925, df_1000, how='left', on=['time','city'])

# kolumna t_delta = podaje rożnice w stopniach 
i_df_delta['t_delta'] = i_df_delta['t_1000']-i_df_delta['t_925']
i_df_delta['inversion_0/1'] = [1 if e > 0 else 0 for e in i_df_delta['t_delta']]

i_df_delta.head()

,time,t_925,city,t_1000,t_delta,inversion_0/1
0,2023-01-01 00:00:00,286.38184,LD,287.68920,1.30736,1
1,2023-01-01 06:00:00,285.93433,LD,287.90503,1.97070,1
2,2023-01-01 12:00:00,286.44385,LD,289.49610,3.05225,1
3,2023-01-01 18:00:00,283.81177,LD,285.81323,2.00146,1
4,2023-01-02 00:00:00,283.38232,LD,283.99048,0.60816,1


In [71]:
# usuwamy nie potrzebne kolumny do dalszej analizy 

i_df_delta = i_df_delta.drop(['t_925','t_1000','t_delta'],axis=1)

In [72]:
# Sprawdzenie poprawnosci inwersja_DataFrame gotowego do megre do głwnego DataFrame

display(i_df_delta.iloc[1000:1010])
print('^' * 140) 
display(i_df_delta['city'].unique())
display(i_df_delta['inversion_0/1'].unique())
print('^' * 140) 
display(i_df_delta.info())

,time,city,inversion_0/1
1000,2023-09-08 00:00:00,LD,1
1001,2023-09-08 06:00:00,LD,1
1002,2023-09-08 12:00:00,LD,1
1003,2023-09-08 18:00:00,LD,1
1004,2023-09-09 00:00:00,LD,1
1005,2023-09-09 06:00:00,LD,1
1006,2023-09-09 12:00:00,LD,1
1007,2023-09-09 18:00:00,LD,1
1008,2023-09-10 00:00:00,LD,1
1009,2023-09-10 06:00:00,LD,1


^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


array(['LD', 'PZ'], dtype=object)

array([1, 0])

^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5848 entries, 0 to 5847
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   time           5848 non-null   datetime64[ns]
 1   city           5848 non-null   object        
 2   inversion_0/1  5848 non-null   int64         
dtypes: datetime64[ns](1), int64(1), object(1)
memory usage: 137.2+ KB


None

#### 1.3.4. PODSUMOWANIE PKT. 1.3. Inwersje
OPIS ZMIENNYCH 
- 'PZ' -> MIASTO ZAPISANE W SYMOBOLEM
    - PZ = Poznań
    - LD = Łódź
- 'time'............ DATA GODZINA
- 'inverion_0/1' -> inwersja_0 = NIE MA / inwerja_1 = JEST

## 1.4. DANE ZANIECZYSZCZEŃ POWIETRZA - [data_polution_2023 , data_polution_2024]

### 1.4.1. Import datasetów. 

In [76]:
polution_2023 = pd.read_csv('/Users/marcinurbanski/Desktop/jmu_repo/1_projekt_air_polution_repo/0_ready_raw_data/data_polution_2023')
polution_2024 = pd.read_csv('/Users/marcinurbanski/Desktop/jmu_repo/1_projekt_air_polution_repo/0_ready_raw_data/data_polution_2024')
# SZYBKIE SPRAWDZENIE 
display(polution_2023.head())
display(polution_2024.head())

,Unnamed: 0,Date_time,LdLodzCzerni_PM10,WpPoznSzwajc_PM10,LdLodzCzerni_PM25,WpPoznSzwajc_PM25,LdLodzCzerni_NO2,WpPoznSzwajc_NO2,LdLodzJanPaw_CO,WpPoznSzwajc_CO
0,0,2023-01-01 01:00:00,25.0,9.87387,21.3840,8.16218,7.9,4.27716,0.394,0.32176
1,1,2023-01-01 02:00:00,7.0,14.97750,6.4152,12.98470,3.8,5.02854,0.430,0.32205
2,2,2023-01-01 03:00:00,7.0,8.45770,6.4152,6.09022,4.3,4.16212,0.410,0.29743
3,3,2023-01-01 04:00:00,7.0,7.40454,5.4351,5.22048,4.0,3.76431,0.401,0.29140
4,4,2023-01-01 05:00:00,6.0,8.40526,5.4351,5.67849,4.4,3.59169,0.379,0.28409


,Unnamed: 0,Date_time,LdLodzCzerni_PM10,WpPoznSzwajc_PM10,LdLodzCzerni_PM25,WpPoznSzwajc_PM25,LdLodzCzerni_NO2,WpPoznSzwajc_NO2,LdLodzJanPaw_CO,WpPoznDabrow_CO
0,0,2024-01-01 01:00:00,48.0,72.3,43.2,67.9,NaN,NaN,0.682,0.621
1,1,2024-01-01 02:00:00,44.0,93.2,39.2,88.7,10.2,19.8,0.783,0.668
2,2,2024-01-01 03:00:00,44.0,87.6,39.2,84.6,12.4,22.0,0.786,0.684
3,3,2024-01-01 04:00:00,45.0,78.5,40.2,75.8,13.3,20.9,0.777,0.576
4,4,2024-01-01 05:00:00,44.0,52.6,40.2,50.8,15.6,11.5,0.703,0.425


#### 1.4.1. Połaczenie obu datasetów d jednego DataFrame

In [78]:
# Połczenie obu lat w jeden chronologiczny dataset

polution_df = pd.concat([polution_2023, polution_2024],  axis=0, ignore_index=True)

#### 1.4.2. Agregeacja danych do postaci uniwersalnego DataFrame

In [80]:
# usuwamy zbedne kolumny, nie potrzebne do analizy i dalszej agregacji 

polution_df = polution_df.drop(['Unnamed: 0'], axis=1) 

In [81]:
type(polution_df['Date_time'][0])

str

In [82]:
# Zmiana typu danych kolumny 'Date_time' ze str na date_time 

polution_df['Date_time'] = pd.to_datetime(polution_df['Date_time'])

# Zamiania nazwy kolumny Date_time na 'time', aby dopasować do pozostałych kolumn z 'time'

polution_df = polution_df.rename(columns={'Date_time' : 'time'})

In [83]:
polution_df.head()

,time,LdLodzCzerni_PM10,WpPoznSzwajc_PM10,LdLodzCzerni_PM25,WpPoznSzwajc_PM25,LdLodzCzerni_NO2,WpPoznSzwajc_NO2,LdLodzJanPaw_CO,WpPoznSzwajc_CO,WpPoznDabrow_CO
0,2023-01-01 01:00:00,25.0,9.87387,21.3840,8.16218,7.9,4.27716,0.394,0.32176,NaN
1,2023-01-01 02:00:00,7.0,14.97750,6.4152,12.98470,3.8,5.02854,0.430,0.32205,NaN
2,2023-01-01 03:00:00,7.0,8.45770,6.4152,6.09022,4.3,4.16212,0.410,0.29743,NaN
3,2023-01-01 04:00:00,7.0,7.40454,5.4351,5.22048,4.0,3.76431,0.401,0.29140,NaN
4,2023-01-01 05:00:00,6.0,8.40526,5.4351,5.67849,4.4,3.59169,0.379,0.28409,NaN


### 2.0. łaczenie danych pogodowych oraz danych całkowitych opadów do jednego DataFrame.
DANE_POGODOWE + DANE_CAŁKOWITYCH OPADÓW + INWERSJA => all_weather_df

In [97]:
# Łaczymy wszyskie DataFrame pogodowe do jednej all_weather_df

all_weather_df = w_df.merge(tp_df, on = ['time','city'], how = 'left').merge(i_df_delta, on = ['time','city'], how = 'left')

In [99]:
all_weather_df.head()

,city,time,u10,v10,v_w,t2m,blh,sp,tcc,tp,inversion_0/1
0,LD,2023-01-01 00:00:00,5.353332,4.853073,7.225682,285.84010,647.28076,99286.125,0.787537,0.0,1
1,LD,2023-01-01 06:00:00,6.153107,3.718613,7.189493,286.22705,745.22437,99435.440,0.596741,0.0,1
2,LD,2023-01-01 12:00:00,5.434403,3.866943,6.669781,289.67798,815.42725,99555.250,0.809631,0.0,1
3,LD,2023-01-01 18:00:00,3.296982,3.080399,4.512089,284.96606,291.85790,99843.500,0.985565,0.0,1
4,LD,2023-01-02 00:00:00,2.058639,3.253723,3.850287,281.79956,184.28812,99938.310,0.989990,0.0,1


In [86]:
w_df.head(3)

,city,time,u10,v10,v_w,t2m,blh,sp,tcc
0,LD,2023-01-01 00:00:00,5.353332,4.853073,7.225682,285.84010,647.28076,99286.125,0.787537
1,LD,2023-01-01 06:00:00,6.153107,3.718613,7.189493,286.22705,745.22437,99435.440,0.596741
2,LD,2023-01-01 12:00:00,5.434403,3.866943,6.669781,289.67798,815.42725,99555.250,0.809631


In [88]:
tp_df.head(3)

,time,tp,city
5,2023-01-01 00:00:00,0.0,LD
11,2023-01-01 06:00:00,0.0,LD
17,2023-01-01 12:00:00,0.0,LD


In [90]:
i_df_delta.head(3)

,time,city,inversion_0/1
0,2023-01-01 00:00:00,LD,1
1,2023-01-01 06:00:00,LD,1
2,2023-01-01 12:00:00,LD,1


In [ ]:
# Rozbicie daty/godziny 'time' na szczegóły i zapisanie w nowych kolumnach 
    #zamiana wartości w kolumnie 'time' na typ datetime
w_ld_2004['time'] = pd.to_datetime(w_ld_2004['time'])
w_ld_2024['time'] = pd.to_datetime(w_ld_2024['time'])
w_pz_2004['time'] = pd.to_datetime(w_pz_2004['time'])
w_pz_2024['time'] = pd.to_datetime(w_pz_2024['time'])

w_ld_2004 = w_ld_2004.assign(
    month=w_ld_2004['time'].dt.month_name(),
    year=w_ld_2004['time'].dt.year,
    day=w_ld_2004['time'].dt.day,
    day_of_week=w_ld_2004['time'].dt.day_name())

w_ld_2024 = w_ld_2024.assign(
    month=w_ld_2024['time'].dt.month_name(),
    year=w_ld_2024['time'].dt.year,
    day=w_ld_2024['time'].dt.day,
    day_of_week=w_ld_2024['time'].dt.day_name())

w_pz_2004 = w_pz_2004.assign(
    month=w_pz_2004['time'].dt.month_name(),
    year=w_pz_2004['time'].dt.year,
    day=w_pz_2004['time'].dt.day,
    day_of_week=w_pz_2004['time'].dt.day_name()) 

w_pz_2024 = w_pz_2024.assign(
    month=w_pz_2024['time'].dt.month_name(),
    year=w_pz_2024['time'].dt.year,
    day=w_pz_2024['time'].dt.day,
    day_of_week=w_pz_2024['time'].dt.day_name()) 